# 01 · La table maître : une ligne par Pokémon

Le but de ce notebook c'est de construire **la table de base du projet** : une ligne = une espèce de Pokémon (repérée par son numéro de Pokédex), avec tous ses attributs (types, stats, taille, poids, génération, stade d'évolution, légendaire, starter…) et les mesures de popularité qui ont déjà un numéro de Pokédex.

Toutes les autres analyses (sur/sous-cotés, conception, réseau, noms) partent de cette table.

On commence par découvrir chaque fichier de PokéAPI avant de les joindre.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 50)
RAW = Path("../data/raw")
OUT = Path("../data/processed")
OUT.mkdir(parents=True, exist_ok=True)

## 1. Les espèces (`pokemon_species.csv`)

C'est le fichier central : une ligne par espèce, avec la génération, l'espèce dont elle évolue, la couleur officielle, la forme du corps et les drapeaux légendaire / fabuleux / bébé.

In [2]:
species = pd.read_csv(RAW / "pokeapi/pokemon_species.csv")
print(species.shape)
species.head()

(1025, 20)


,id,identifier,generation_id,evolves_from_species_id,evolution_chain_id,color_id,shape_id,habitat_id,gender_rate,capture_rate,base_happiness,is_baby,hatch_counter,has_gender_differences,growth_rate_id,forms_switchable,is_legendary,is_mythical,order,conquest_order
0,1,bulbasaur,1,NaN,1,5,8,3.0,1,45,70,0,20,0,4,0,0,0,1,NaN
1,2,ivysaur,1,1.0,1,5,8,3.0,1,45,70,0,20,0,4,0,0,0,2,NaN
2,3,venusaur,1,2.0,1,5,8,3.0,1,45,70,0,20,1,4,1,0,0,3,NaN
3,4,charmander,1,NaN,2,8,6,4.0,1,45,70,0,20,0,4,0,0,0,4,109.0
4,5,charmeleon,1,4.0,2,8,6,4.0,1,45,70,0,20,0,4,0,0,0,5,110.0


In [3]:
species.dtypes

id                           int64
identifier                     str
generation_id                int64
evolves_from_species_id    float64
evolution_chain_id           int64
color_id                     int64
shape_id                     int64
habitat_id                 float64
gender_rate                  int64
capture_rate                 int64
base_happiness               int64
is_baby                      int64
hatch_counter                int64
has_gender_differences       int64
growth_rate_id               int64
forms_switchable             int64
is_legendary                 int64
is_mythical                  int64
order                        int64
conquest_order             float64
dtype: object

In [4]:
species.isna().sum()

id                           0
identifier                   0
generation_id                0
evolves_from_species_id    541
evolution_chain_id           0
color_id                     0
shape_id                     0
habitat_id                 639
gender_rate                  0
capture_rate                 0
base_happiness               0
is_baby                      0
hatch_counter                0
has_gender_differences       0
growth_rate_id               0
forms_switchable             0
is_legendary                 0
is_mythical                  0
order                        0
conquest_order             825
dtype: int64

On a bien **1 025 espèces**, une par ligne. Les seuls manquants sont normaux : `evolves_from_species_id` est vide pour les 541 Pokémon qui n'évoluent de rien, `habitat_id` n'existe que pour les vieilles générations et `conquest_order` concerne un spin-off. On n'utilisera ni l'habitat ni `conquest_order`.

In [5]:
# combien de Pokémon par génération ?
species["generation_id"].value_counts().sort_index()

generation_id
1    151
2    100
3    135
4    107
5    156
6     72
7     88
8     96
9    120
Name: count, dtype: int64

Les générations n'ont pas du tout la même taille : de 72 Pokémon (génération 6) à 156 (génération 5). Il faudra s'en souvenir quand on compare les générations entre elles.

In [6]:
# légendaires, fabuleux, bébés : combien, et est-ce qu'ils se chevauchent ?
print(species[["is_legendary", "is_mythical", "is_baby"]].sum())
print("légendaire ET fabuleux :", ((species.is_legendary == 1) & (species.is_mythical == 1)).sum())

is_legendary    71
is_mythical     23
is_baby         19
dtype: int64
légendaire ET fabuleux : 0


71 légendaires, 23 fabuleux et 19 bébés. Un Pokémon n'est jamais légendaire et fabuleux en même temps, ce sont deux catégories séparées.

## 2. Les formes (`pokemon.csv`)

La taille et le poids sont rangés par **forme** et pas par espèce : une espèce peut avoir plusieurs formes (méga-évolutions, formes d'Alola, de Galar…). Il faut regarder ce que ça donne avant de choisir laquelle garder.

In [7]:
forms = pd.read_csv(RAW / "pokeapi/pokemon.csv")
print(forms.shape)
print(forms["is_default"].value_counts())
forms.head()

(1351, 8)
is_default
1    1025
0     326
Name: count, dtype: int64


,id,identifier,species_id,height,weight,base_experience,order,is_default
0,1,bulbasaur,1,7,69,64.0,1.0,1
1,2,ivysaur,2,10,130,142.0,2.0,1
2,3,venusaur,3,20,1000,236.0,3.0,1
3,4,charmander,4,6,85,62.0,5.0,1
4,5,charmeleon,5,11,190,142.0,6.0,1


In [8]:
# est-ce qu'il y a bien exactement une forme par défaut par espèce ?
forms[forms.is_default == 1].groupby("species_id").size().value_counts()

1    1025
Name: count, dtype: int64

In [9]:
# quelques exemples de formes qui ne sont pas par défaut
forms[forms.is_default == 0].sample(10, random_state=0)[["identifier", "species_id", "height", "weight"]]

,identifier,species_id,height,weight
1147,oricorio-pom-pom,741,6,34
1091,altaria-mega,334,15,206
1167,mimikyu-busted,778,2,7
1270,palkia-origin,484,63,6590
1171,magearna-original,801,10,805
1088,swampert-mega,260,19,1020
1294,miraidon-aquatic-mode,1008,28,2400
1040,basculin-blue-striped,550,10,180
1161,minior-orange,774,3,3
1169,mimikyu-totem-busted,778,4,28


Les formes qui ne sont pas par défaut, ce sont les méga-évolutions, les formes régionales, les variantes de couleur (Minior), les formes de combat (Mimikyu)… En gros ce sont des variantes d'un même Pokémon, pas des Pokémon différents. Chaque espèce a **exactement une** forme par défaut.

Du coup on décide de **ne garder que la forme par défaut** pour la taille, le poids, les types et les stats. Les sondages votent pour un Pokémon, pas pour sa méga-évolution, donc c'est la forme « normale » qui a du sens ici, et ça évite d'avoir plusieurs lignes pour un même Pokémon.

In [10]:
# la taille est en décimètres et le poids en hectogrammes : on regarde les extrêmes
default = forms[forms.is_default == 1]
default[["height", "weight"]].describe()

,height,weight
count,1025.000000,1025.000000
mean,12.116098,669.818537
std,12.481673,1212.754122
min,1.000000,1.000000
25%,5.000000,85.000000
50%,10.000000,280.000000
75%,15.000000,700.000000
max,200.000000,9999.000000


In [11]:
default.sort_values("weight", ascending=False).head(5)[["identifier", "height", "weight"]]

,identifier,height,weight
789,cosmoem,1,9999
796,celesteela,92,9999
382,groudon,35,9500
889,eternatus,200,9500
749,mudsdale,25,9200


Attention aux unités : la taille est en décimètres et le poids en hectogrammes (on divisera par 10 pour avoir des mètres et des kilos). Le poids est très asymétrique (médiane 28 kg, mais jusqu'à 999,9 kg). Les 999,9 kg de Cosmovum et Bamboiselle sont les vraies valeurs du jeu (c'est le maximum possible), pas une erreur de saisie.

## 3. Les types (`pokemon_types.csv` + `types.csv`)

Chaque forme a un ou deux types, rangés par `slot` (1 = type principal, 2 = type secondaire).

In [12]:
ptypes = pd.read_csv(RAW / "pokeapi/pokemon_types.csv")
types = pd.read_csv(RAW / "pokeapi/types.csv")
print(types[["id", "identifier"]].to_string(index=False))
ptypes.head()

   id identifier
    1     normal
    2   fighting
    3     flying
    4     poison
    5     ground
    6       rock
    7        bug
    8      ghost
    9      steel
   10       fire
   11      water
   12      grass
   13   electric
   14    psychic
   15        ice
   16     dragon
   17       dark
   18      fairy
   19    stellar
10001    unknown
10002     shadow


,pokemon_id,type_id,slot
0,1,12,1
1,1,4,2
2,2,12,1
3,2,4,2
4,3,12,1


In [13]:
ptypes = ptypes.merge(types[["id", "identifier"]], left_on="type_id", right_on="id")
types_wide = ptypes.pivot(index="pokemon_id", columns="slot", values="identifier")
types_wide.columns = ["type1", "type2"]
types_wide.head()

,type1,type2
pokemon_id,,
1,grass,poison
2,grass,poison
3,grass,poison
4,fire,NaN
5,fire,NaN


Les types sont en anglais. `stellar`, `unknown` et `shadow` sont des types spéciaux qui ne sont portés par aucune forme normale. Un peu moins de la moitié des Pokémon n'ont qu'un seul type (le `type2` vide n'est donc pas un manquant, c'est juste « pas de second type »).

## 4. Les statistiques de combat (`pokemon_stats.csv` + `stats.csv`)

In [14]:
pstats = pd.read_csv(RAW / "pokeapi/pokemon_stats.csv")
stats = pd.read_csv(RAW / "pokeapi/stats.csv")
print(stats[["id", "identifier"]].to_string(index=False))
pstats.groupby("pokemon_id").size().value_counts()

 id      identifier
  1              hp
  2          attack
  3         defense
  4  special-attack
  5 special-defense
  6           speed
  7        accuracy
  8         evasion
  9         special


6    1351
Name: count, dtype: int64

In [15]:
pstats = pstats.merge(stats[["id", "identifier"]], left_on="stat_id", right_on="id")
stats_wide = pstats.pivot(index="pokemon_id", columns="identifier", values="base_stat")
stats_wide = stats_wide[["hp", "attack", "defense", "special-attack", "special-defense", "speed"]]
stats_wide.columns = ["pv", "attaque", "defense", "attaque_spe", "defense_spe", "vitesse"]
stats_wide["total_stats"] = stats_wide.sum(axis=1)
stats_wide.describe().T

,count,mean,std,min,25%,50%,75%,max
pv,1351.0,71.673575,26.827529,1.0,55.0,70.0,85.0,255.0
attaque,1351.0,82.757957,32.351906,5.0,60.0,80.0,102.5,190.0
defense,1351.0,76.193930,30.712902,5.0,55.0,70.0,95.0,250.0
attaque_spe,1351.0,75.418209,34.026272,10.0,50.0,69.0,97.0,216.0
defense_spe,1351.0,74.115470,27.800826,20.0,54.0,70.0,90.0,250.0
vitesse,1351.0,72.000000,30.642043,5.0,49.0,70.0,95.0,200.0
total_stats,1351.0,452.159141,122.456241,175.0,340.0,475.0,530.0,1125.0


Les 6 stats sont présentes pour toutes les formes (`accuracy`, `evasion` et `special` ne sont pas des stats de base). Le total des stats va de 175 à 1 125 (le maximum vient d'une forme spéciale ; on ne garde de toute façon que les formes par défaut).

## 5. Les noms (`pokemon_species_names.csv`)

On garde l'anglais (clé pour Google Trends et les sondages), le français, et le japonais en kana et en romaji (pour l'axe Noms).

In [16]:
names = pd.read_csv(RAW / "pokeapi/pokemon_species_names.csv")
langs = pd.read_csv(RAW / "pokeapi/languages.csv")
langs[["id", "identifier"]]

,id,identifier
0,1,ja-hrkt
1,2,ja-roma
2,3,ko
3,4,zh-hant
4,5,fr
5,6,de
6,7,es
7,8,it
8,9,en
9,10,cs


In [17]:
keep = {9: "nom_en", 5: "nom_fr", 1: "nom_ja", 2: "nom_ja_roma"}
names_wide = (names[names.local_language_id.isin(keep)]
              .pivot(index="pokemon_species_id", columns="local_language_id", values="name")
              .rename(columns=keep)[list(keep.values())])
names_wide.head()

local_language_id,nom_en,nom_fr,nom_ja,nom_ja_roma
pokemon_species_id,,,,
1,Bulbasaur,Bulbizarre,フシギダネ,Fushigidane
2,Ivysaur,Herbizarre,フシギソウ,Fushigisou
3,Venusaur,Florizarre,フシギバナ,Fushigibana
4,Charmander,Salamèche,ヒトカゲ,Hitokage
5,Charmeleon,Reptincel,リザード,Lizardo


In [18]:
names_wide.isna().sum()

local_language_id
nom_en         0
nom_fr         0
nom_ja         0
nom_ja_roma    0
dtype: int64

Les 4 noms sont présents pour les 1 025 espèces, aucun manquant.

## 6. Le stade d'évolution

Il n'y a pas de colonne « stade » toute faite. On le déduit de `evolves_from_species_id` : un Pokémon qui n'évolue de rien est au stade 1, celui qui évolue d'un stade 1 est au stade 2, etc.

On calcule aussi si le Pokémon est **la dernière évolution** de sa lignée (personne n'évolue de lui).

In [19]:
parent = species.set_index("id")["evolves_from_species_id"]

def stade(i):
    s = 1
    while pd.notna(parent[i]):
        i = int(parent[i])
        s += 1
    return s

species["stade"] = species["id"].map(stade)
a_des_enfants = set(species["evolves_from_species_id"].dropna().astype(int))
species["evolution_finale"] = ~species["id"].isin(a_des_enfants)
species["stade"].value_counts().sort_index()

stade
1    541
2    363
3    121
Name: count, dtype: int64

541 Pokémon au stade 1, 363 au stade 2 et 121 au stade 3.

In [20]:
# le cas des bébés : ils ont été ajoutés après coup devant des lignées existantes
noms = names_wide["nom_fr"]
bebes = species[species.is_baby == 1]
pd.DataFrame({
    "bebe": bebes["id"].map(noms).values,
    "stade_bebe": bebes["stade"].values,
    "evolue_en": [noms[species.loc[species.evolves_from_species_id == i, "id"]].tolist() for i in bebes["id"]],
})

,bebe,stade_bebe,evolue_en
0,Pichu,1,[Pikachu]
1,Mélo,1,[Mélofée]
2,Toudoudou,1,[Rondoudou]
3,Togepi,1,[Togetic]
4,Debugant,1,"[Kicklee, Tygnon, Kapoera]"
5,Lippouti,1,[Lippoutou]
6,Élekid,1,[Élektek]
7,Magby,1,[Magmar]
8,Azurill,1,[Marill]
9,Okéoké,1,[Qulbutoké]


Le problème des **bébés** : ils ont été ajoutés dans les générations 2 et 4 devant des lignées qui existaient déjà. Du coup Pikachu passe au stade 2 parce que Pichu existe, alors que Pikachu a été créé comme un Pokémon de base. Pareil pour Ronflex, Leveinard, Lucario, etc.

On a hésité à mettre les bébés à un « stade 0 » pour que Pikachu reste au stade 1. Finalement **on garde la lignée telle qu'elle est aujourd'hui** (Pichu stade 1, Pikachu stade 2, Raichu stade 3) : c'est comme ça que les joueurs actuels voient la lignée, et ça évite d'inventer une règle à part pour 19 Pokémon.

In [21]:
# les Pokémon qui n'ont aucune évolution (ni avant ni après)
seul = (species["stade"] == 1) & species["evolution_finale"]
print("Pokémon sans aucune évolution :", seul.sum())
print("dont légendaires ou fabuleux :", (seul & ((species.is_legendary == 1) | (species.is_mythical == 1))).sum())

Pokémon sans aucune évolution : 201
dont légendaires ou fabuleux : 84


201 Pokémon n'ont aucune évolution, et parmi eux 84 sont légendaires ou fabuleux. Pour eux le « stade 1 » ne veut pas dire « Pokémon de base faible » : un Mewtwo ou un Lokhlass n'a rien à voir avec un Chenipan.

Du coup on décide de les mettre dans **une catégorie à part, « sans évolution »**. Sinon ils seraient mélangés avec les vrais stades 1 et ils brouilleraient toutes les comparaisons par stade.

On crée donc une colonne `stade_categorie` : « sans évolution » pour ces 201 Pokémon, et « stade 1 », « stade 2 », « stade 3 » pour les autres. On garde quand même la colonne `stade` en nombre, ça peut servir pour des corrélations.

In [22]:
species["stade_categorie"] = "stade " + species["stade"].astype(str)
species.loc[seul, "stade_categorie"] = "sans évolution"
species["stade_categorie"].value_counts()

stade_categorie
stade 2           363
stade 1           340
sans évolution    201
stade 3           121
Name: count, dtype: int64

## 7. Les starters (`starters.csv`)

Fichier construit à la main à partir de Bulbapedia : les 27 lignées de starters, avec leurs évolutions.

In [23]:
starters = pd.read_csv(RAW / "starters/starters.csv")
print(starters.shape)
starters.head(6)

(81, 7)


,dex,nom_en,nom_fr,generation,type_starter,dex_starter,stade
0,1,Bulbasaur,Bulbizarre,1,Plante,1,1
1,2,Ivysaur,Herbizarre,1,Plante,1,2
2,3,Venusaur,Florizarre,1,Plante,1,3
3,4,Charmander,Salamèche,1,Feu,4,1
4,5,Charmeleon,Reptincel,1,Feu,4,2
5,6,Charizard,Dracaufeu,1,Feu,4,3


In [24]:
starters.groupby(['generation', 'type_starter']).size().unstack()

type_starter,Eau,Feu,Plante
generation,,,
1,3,3,3
2,3,3,3
3,3,3,3
4,3,3,3
5,3,3,3
6,3,3,3
7,3,3,3
8,3,3,3
9,3,3,3


On a bien les 27 lignées de starters (3 par génération et par type), soit 81 Pokémon.

## 8. Les sources de popularité qui ont déjà un numéro de Pokédex

Avant de les ajouter, un choix important : **on ne fait pas de score de popularité combiné**. Chaque source garde sa propre colonne. Un Pokémon peut être très aimé chez un public (ou dans un pays) et beaucoup moins ailleurs, et les sources n'ont pas du tout le même nombre de votants (52 000 sur Reddit, plus de 500 000 au Johto Times, des millions de recherches Google). Une moyenne ferait disparaître ces différences, alors que c'est justement intéressant. La façon de comparer les sources entre elles est faite dans le notebook 02.

- **Reddit 2019** : le fichier n'a pas de colonne « numéro », mais l'index ressemble à un numéro de Pokédex. Il faut le vérifier en comparant les noms avec PokéAPI.
- **Johto Times 2025** : il a une colonne `dex`, on vérifie aussi les noms.
- **Pokémon of the Year 2020** : il n'a que des noms, l'harmonisation des noms est faite par Lucas, donc on ne le joint pas ici.

In [25]:
reddit = pd.read_csv(RAW / "reddit_2019/df_ranked.csv", index_col=0)
print(reddit.shape, "index unique :", reddit.index.is_unique, "| de", reddit.index.min(), "à", reddit.index.max())
reddit.head()

(809, 9) index unique : True | de 1 à 809


,name,votes,types,generation,family,sprite_source,generation_color,ranking_overall,ranking_generation
734,Yungoos,0.0,Normal,7,Yungoos,https://raw.githubusercontent.com/PokeAPI/spri...,#E89483,809,88.0
266,Silcoon,0.0,Bug,3,Wurmple,https://raw.githubusercontent.com/PokeAPI/spri...,#9CD7C8,808,135.0
603,Eelektrik,0.0,Electric,5,Tynamo,https://raw.githubusercontent.com/PokeAPI/spri...,#9FCADF,807,155.0
574,Gothita,0.0,Psychic,5,Gothita,https://raw.githubusercontent.com/PokeAPI/spri...,#9FCADF,806,156.0
343,Baltoy,1.0,Ground Psychic,3,Baltoy,https://raw.githubusercontent.com/PokeAPI/spri...,#9CD7C8,805,134.0


In [26]:
# est-ce que l'index est bien le numéro de Pokédex ? on compare les noms
comp = pd.DataFrame({"reddit": reddit["name"], "pokeapi": names_wide.loc[reddit.index, "nom_en"]})
diff = comp[comp.reddit.str.lower() != comp.pokeapi.str.lower()]
print(len(diff), "noms différents")
diff

11 noms différents


,reddit,pokeapi
788,Tapu-Fini,Tapu Fini
787,Tapu-Bulu,Tapu Bulu
786,Tapu-Lele,Tapu Lele
669,Flabebe,Flabébé
29,Nidoran-F,Nidoran♀
439,Mime-Jr,Mime Jr.
772,Type-Null,Type: Null
785,Tapu-Koko,Tapu Koko
32,Nidoran-M,Nidoran♂
122,Mr-Mime,Mr. Mime


L'index de Reddit est bien le numéro de Pokédex : les 11 noms qui diffèrent sont juste écrits autrement (tirets à la place des espaces, ♀/♂ écrits F/M, accents et apostrophes enlevés). Ce sont les mêmes Pokémon.

In [27]:
reddit["votes"].describe()

count     809.000000
mean       65.173053
std       118.625512
min         0.000000
25%         9.000000
50%        23.000000
75%        62.000000
max      1107.000000
Name: votes, dtype: float64

Les votes Reddit sont très concentrés : la moitié des Pokémon ont 23 votes ou moins, alors que le premier en a 1 107. Le total fait 52 725 votes, un peu moins que les 52 836 réponses du sondage (sans doute des réponses qui ne correspondaient à aucun Pokémon).

In [28]:
johto = pd.read_csv(RAW / "johto_times_2025/johto_times_2025_preliminaire.csv")
print(johto.shape, "dex unique :", johto.dex.is_unique)
comp = johto.set_index("dex")[["name_en"]].join(names_wide["nom_en"])
diff = comp[comp.name_en.str.lower() != comp.nom_en.str.lower()]
print(len(diff), "noms différents")
diff

(1025, 10) dex unique : True
0 noms différents


,name_en,nom_en
dex,,


Pour le Johto Times, les numéros, les noms et les générations collent parfaitement avec PokéAPI.

In [29]:
# la génération donnée par le Johto Times correspond-elle à celle de PokéAPI ?
g = johto.set_index("dex")["generation"]
(g != species.set_index("id")["generation_id"]).sum()

np.int64(0)

### Pokémon of the Year 2020 (partie Lucas)

Le vote officiel de 2020 ne publie que **le top 30 général et le top 30 de chacune des 8 régions** de jeu, soit 240 Pokémon différents. On a deux fichiers :

- le fichier Excel repris de Will Sutton (`pokemon_of_the_year_2020.xlsx`), **sans numéro de Pokédex** ;
- les mêmes listes extraites directement du site officiel (`poty_2020_officiel.csv`, script `download_poty_2020.py`), **avec le numéro de Pokédex**.

On vérifie d'abord que les deux disent la même chose.

In [30]:
poty_off = pd.read_csv(RAW / "pokemon_of_the_year_2020/poty_2020_officiel.csv")
poty_xlsx = pd.read_excel(RAW / "pokemon_of_the_year_2020/pokemon_of_the_year_2020.xlsx", sheet_name="pokemon_of_the_year_2020")
print(poty_off.shape, poty_xlsx.shape)
poty_off["liste"].value_counts()

(270, 9) (270, 5)


liste
All       30
Kanto     30
Johto     30
Hoenn     30
Sinnoh    30
Unova     30
Kalos     30
Alola     30
Galar     30
Name: count, dtype: int64

In [31]:
# on compare ligne à ligne (même liste, même rang) ; dans l'Excel, la région Alola est écrite « Aloha »
x = poty_xlsx.rename(columns={"region": "liste", "rank": "rang"}).replace({"liste": {"Aloha": "Alola"}})
comp = x.merge(poty_off, on=["liste", "rang"], suffixes=("_xlsx", "_officiel"))
print(len(comp), "lignes comparées")
comp[(comp.name != comp.nom_en) | (comp.votes_xlsx != comp.votes_officiel)][["liste", "rang", "name", "votes_xlsx", "nom_en", "votes_officiel"]]

270 lignes comparées


,liste,rang,name,votes_xlsx,nom_en,votes_officiel
3,Hoenn,1,Rayquaza,60596,Rayquaza,60939


Les deux fichiers sont identiques **sauf une ligne** : dans l'Excel, Rayquaza a 60 596 votes dans la liste de Hoenn, alors que le site officiel donne 60 939 dans les deux listes (Hoenn et générale). C'est une faute de saisie de l'Excel : **on prend les valeurs du site officiel**.

On joint par **numéro de Pokédex** (donné par le site) plutôt que par le nom : par le nom, Sirfetch'd ne colle pas (apostrophe droite dans l'Excel, apostrophe typographique dans PokéAPI).

In [32]:
# un même Pokémon apparaît dans sa liste régionale, et aussi dans la liste générale s'il est dans le top 30 mondial
regional = poty_off[poty_off.liste != "All"]
print("Pokémon :", regional.dex.nunique(), "| un seul passage par liste régionale :", regional.dex.is_unique)
print("tous les Pokémon du top 30 mondial sont dans une liste régionale :", set(poty_off.loc[poty_off.liste == "All", "dex"]) <= set(regional.dex))

# même nombre de votes dans les deux listes ?
votes_par_dex = poty_off.groupby("dex")["votes"].nunique()
print("Pokémon avec deux nombres de votes différents :", (votes_par_dex > 1).sum())

Pokémon : 240 | un seul passage par liste régionale : True
tous les Pokémon du top 30 mondial sont dans une liste régionale : True
Pokémon avec deux nombres de votes différents : 0


In [33]:
# les listes du site suivent-elles les générations de PokéAPI ?
gen_liste = {"Kanto": 1, "Johto": 2, "Hoenn": 3, "Sinnoh": 4, "Unova": 5, "Kalos": 6, "Alola": 7, "Galar": 8}
r = regional.assign(gen_liste=regional.liste.map(gen_liste))
r = r.join(species.set_index("id")["generation_id"], on="dex")
r[r.gen_liste != r.generation_id][["liste", "rang", "dex", "nom_en", "generation_id"]]

,liste,rang,dex,nom_en,generation_id
254,Galar,15,808,Meltan,7
262,Galar,23,809,Melmetal,7


Seuls **Meltan et Melmetal** sont classés dans la liste Galar alors qu'ils sont de génération 7 dans PokéAPI (ils sont sortis entre les deux générations, avec Pokémon GO). On garde la génération de PokéAPI, comme pour les autres sources.

**Ce qu'on garde dans la table** (une ligne par Pokémon) :

- `poty_votes` : les votes. Le vote est mondial, donc les votes se comparent d'une région à l'autre ;
- `poty_rang_generation` : le rang **dans sa génération** (celle de PokéAPI), calculé à partir des votes. On ne garde pas le rang dans la liste du site : à cause de Meltan et Melmetal, la génération 7 aurait deux 15e et deux 23e, et la génération 8 ni 15e ni 23e. Comme les votes sont mondiaux, on peut reclasser les Pokémon à l'intérieur de chaque génération (on vérifie juste après que ce rang est exact) ;
- `poty_top30_mondial` : le Pokémon est-il dans le top 30 général ?

**Les 785 autres Pokémon restent vides, pas à 0**, mais pour deux raisons différentes :

- **650 Pokémon pouvaient être votés** (n° 1 à 890, tous les Pokémon sortis jusqu'à Épée et Bouclier) : ils ont eu des votes, on sait seulement qu'ils sont derrière le 30e de leur région ;
- **135 Pokémon ne pouvaient pas être votés** (n° 891 à 1025) : ils n'existaient pas encore en février 2020. Ce sont les 15 derniers de la génération 8 (DLC d'Épée et Bouclier, Légendes Arceus) et toute la génération 9. Pour eux, la source n'existe pas.

Les rangs restent exacts : en génération 8, les 28 Pokémon publiés sont les 28 premiers des **81 Pokémon votables** de la génération.

Comme les votes des Pokémon non publiés sont inconnus, **on ne calcule pas de rang mondial** au-delà du top 30 : le 30e de Kalos a 6 293 votes, alors qu'un Pokémon de Kanto absent de la liste peut en avoir presque autant que le 30e de Kanto (9 955).

In [34]:
poty = regional.set_index("dex")[["votes"]].rename(columns={"votes": "poty_votes"})
poty = poty.join(species.set_index("id")["generation_id"])
# rang dans la génération de PokéAPI : 1 = le plus de votes (ex aequo : même rang)
poty["poty_rang_generation"] = poty.groupby("generation_id")["poty_votes"].rank(ascending=False, method="min").astype(int)
poty["poty_top30_mondial"] = poty.index.isin(poty_off.loc[poty_off.liste == "All", "dex"])
print(poty.shape, "| top 30 mondial :", poty.poty_top30_mondial.sum())
poty.groupby("generation_id")["poty_rang_generation"].agg(["count", "min", "max"])


(240, 4) | top 30 mondial : 30


,count,min,max
generation_id,,,
1,30,1,30
2,30,1,30
3,30,1,30
4,30,1,30
5,30,1,30
6,30,1,30
7,32,1,32
8,28,1,28


**Ce rang est-il exact ?** Les Pokémon non publiés d'une génération ont moins de votes que le 30e de la liste de leur région (Alola pour la génération 7, Galar pour la 8…). Le rang dans la génération est donc exact si **tous les Pokémon publiés de la génération ont au moins autant de votes que ce 30e** : dans ce cas, aucun Pokémon non publié ne peut passer devant eux.

In [35]:
seuil = poty_off[poty_off.rang == 30].set_index("liste")["votes"]          # votes du 30e de chaque liste
seuil.index = seuil.index.map(gen_liste)                                    # liste -> génération
verif = poty.groupby("generation_id")["poty_votes"].min().to_frame("votes_du_dernier_publie")
verif["votes_du_30e_de_la_liste"] = seuil
verif["rang_exact"] = verif.votes_du_dernier_publie >= verif.votes_du_30e_de_la_liste
verif

,votes_du_dernier_publie,votes_du_30e_de_la_liste,rang_exact
generation_id,,,
1,9955,9955,True
2,7418,7418,True
3,6459,6459,True
4,7040,7040,True
5,7905,7905,True
6,6293,6293,True
7,7126,7126,True
8,8436,8436,True


Le rang est exact dans les 8 générations : à chaque fois, le dernier Pokémon publié de la génération est justement le 30e de la liste de sa région. En génération 7, Meltan (16 077 votes) et Melmetal (12 356 votes) sont bien au-dessus du 30e d'Alola (7 126) : ils prennent leur place parmi les premiers de la génération 7, qui compte donc 32 Pokémon classés (de 1 à 32), et la génération 8 en compte 28.

## 9. La jointure

On part des espèces et on ajoute tout le reste par le numéro de Pokédex. Pour la taille, le poids, les types et les stats, on prend **la forme par défaut** de chaque espèce.

In [36]:
colors = pd.read_csv(RAW / "pokeapi/pokemon_colors.csv").set_index("id")["identifier"]
shapes = pd.read_csv(RAW / "pokeapi/pokemon_shapes.csv").set_index("id")["identifier"]

base = species.rename(columns={"id": "dex", "generation_id": "generation"})
table = pd.DataFrame({
    "dex": base["dex"],
    "generation": base["generation"],
    "couleur": base["color_id"].map(colors),
    "forme_corps": base["shape_id"].map(shapes),
    "legendaire": base["is_legendary"].astype(bool),
    "fabuleux": base["is_mythical"].astype(bool),
    "bebe": base["is_baby"].astype(bool),
    "stade": base["stade"],
    "stade_categorie": base["stade_categorie"],
    "evolution_finale": base["evolution_finale"],
    "taux_capture": base["capture_rate"],
}).set_index("dex")

table = names_wide.join(table)

# forme par défaut : on passe de l'id de forme à l'id d'espèce
d = default.set_index("id")
phys = pd.DataFrame({"taille_m": d["height"] / 10, "poids_kg": d["weight"] / 10}, index=d.index)
phys = phys.join(types_wide).join(stats_wide)
phys.index = d.loc[phys.index, "species_id"]
table = table.join(phys)

# starters
st = starters.set_index("dex")[["type_starter", "dex_starter"]]
table = table.join(st)
table["starter"] = table["type_starter"].notna()

# popularité
table["reddit_votes"] = reddit["votes"]
table["johto_pct_generation"] = johto.set_index("dex")["pct_generation"]
table["johto_rang_generation"] = johto.set_index("dex")["rang_generation"]
table = table.join(poty.drop(columns="generation_id"))
# vrai/faux seulement pour les 240 publiés : vide pour les autres, comme les votes
table["poty_top30_mondial"] = table["poty_top30_mondial"].astype("boolean")

table.index.name = "dex"
print(table.shape)
table.head()

(1025, 34)


,nom_en,nom_fr,nom_ja,nom_ja_roma,generation,couleur,forme_corps,legendaire,fabuleux,bebe,stade,stade_categorie,evolution_finale,taux_capture,taille_m,poids_kg,type1,type2,pv,attaque,defense,attaque_spe,defense_spe,vitesse,total_stats,type_starter,dex_starter,starter,reddit_votes,johto_pct_generation,johto_rang_generation,poty_votes,poty_rang_generation,poty_top30_mondial
dex,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,Bulbasaur,Bulbizarre,フシギダネ,Fushigidane,1,green,quadruped,False,False,False,1,stade 1,False,45,0.7,6.9,grass,poison,45,49,49,65,65,45,318,Plante,1.0,True,710.0,2.93,7,56015.0,3.0,True
2,Ivysaur,Herbizarre,フシギソウ,Fushigisou,1,green,quadruped,False,False,False,2,stade 2,False,45,1.0,13.0,grass,poison,60,62,63,80,80,60,405,Plante,1.0,True,83.0,0.63,50,NaN,NaN,<NA>
3,Venusaur,Florizarre,フシギバナ,Fushigibana,1,green,quadruped,False,False,False,3,stade 3,True,45,2.0,100.0,grass,poison,80,82,83,100,100,80,525,Plante,1.0,True,127.0,1.35,24,10454.0,28.0,False
4,Charmander,Salamèche,ヒトカゲ,Hitokage,1,red,upright,False,False,False,1,stade 1,False,45,0.6,8.5,fire,NaN,39,52,43,60,50,65,309,Feu,4.0,True,374.0,0.77,37,14049.0,17.0,False
5,Charmeleon,Reptincel,リザード,Lizardo,1,red,upright,False,False,False,2,stade 2,False,45,1.1,19.0,fire,NaN,58,64,58,80,65,80,405,Feu,4.0,True,70.0,0.25,89,NaN,NaN,<NA>


## 10. Contrôles de la jointure

Juste les contrôles qui disent si la jointure s'est bien passée (les contrôles de qualité complets, c'est la partie de Lucas : voir la section 11).

In [37]:
print("lignes :", len(table), "| dex unique :", table.index.is_unique)
table.isna().sum()[lambda s: s > 0]

lignes : 1025 | dex unique : True


type2                   499
type_starter            944
dex_starter             944
reddit_votes            216
poty_votes              785
poty_rang_generation    785
poty_top30_mondial      785
dtype: int64

In [38]:
# les manquants de Reddit doivent être exactement les générations 8 et 9
table.loc[table.reddit_votes.isna(), "generation"].value_counts().sort_index()

generation
8     96
9    120
Name: count, dtype: int64

La jointure s'est bien passée : 1 025 lignes, un numéro unique par ligne. Les seuls manquants sont attendus : pas de second type (499), pas starter (944), et Reddit qui ne couvre pas les générations 8 et 9 (216 = 96 + 120).

In [39]:
# Pokémon of the Year 2020 : exactement les 240 Pokémon publiés, et aucun 0
print("Pokémon avec des votes POTY :", table.poty_votes.notna().sum(), "| minimum :", table.poty_votes.min())
table.groupby("generation")["poty_votes"].count()

Pokémon avec des votes POTY : 240 | minimum : 6293.0


generation
1    30
2    30
3    30
4    30
5    30
6    30
7    32
8    28
9     0
Name: poty_votes, dtype: int64

Pour Pokémon of the Year 2020, on a bien les 240 Pokémon publiés : 30 par génération de 1 à 6, 32 en génération 7 et 28 en génération 8 (Meltan et Melmetal, classés dans la liste Galar, comptent en génération 7 comme dans PokéAPI), et aucun en génération 9. Seuls les n° 1 à 890 pouvaient être votés : la génération 9 et les 15 derniers Pokémon de la génération 8 (n° 891 à 905) sont sortis après le vote.

In [40]:
table.dtypes

nom_en                       str
nom_fr                       str
nom_ja                       str
nom_ja_roma                  str
generation                 int64
couleur                      str
forme_corps                  str
legendaire                  bool
fabuleux                    bool
bebe                        bool
stade                      int64
stade_categorie              str
evolution_finale            bool
taux_capture               int64
taille_m                 float64
poids_kg                 float64
type1                        str
type2                        str
pv                         int64
attaque                    int64
defense                    int64
attaque_spe                int64
defense_spe                int64
vitesse                    int64
total_stats                int64
type_starter                 str
dex_starter              float64
starter                     bool
reddit_votes             float64
johto_pct_generation     float64
johto_rang

## 11. Contrôles de qualité (partie Lucas)

La section 10 vérifie que la jointure s'est bien passée. Ici, on contrôle la **qualité** de la table elle-même, avant de l'exporter, en suivant la démarche du TP1 :

1. les doublons ;
2. les manquants ;
3. la cohérence entre colonnes ;
4. les types de variables ;
5. les valeurs extrêmes ;
6. un cas particulier : le poids plafonné ;
7. la mise à l'échelle.

### 11.1 Les doublons

In [41]:
print("numéros de Pokédex en double :", table.index.duplicated().sum())
for col in ["nom_en", "nom_fr", "nom_ja", "nom_ja_roma"]:
    print(f"{col:12} en double : {table[col].duplicated().sum()}")
# une seule forme par espèce : on a gardé la forme par défaut (section 2)
print("formes par défaut par espèce :", default.groupby("species_id").size().unique())

numéros de Pokédex en double : 0
nom_en       en double : 0
nom_fr       en double : 0
nom_ja       en double : 0
nom_ja_roma  en double : 0
formes par défaut par espèce : [1]


Aucun doublon : chaque numéro de Pokédex et chaque nom (anglais, français, japonais, romaji) n'apparaît qu'une fois, et on a gardé exactement une forme par espèce.

### 11.2 Les manquants

In [42]:
manquants = table.isna().sum()[lambda s: s > 0].to_frame("manquants")
manquants["explication"] = manquants.index.map({
    "type2": "Pokémon à un seul type",
    "type_starter": "pas un starter",
    "dex_starter": "pas un starter",
    "reddit_votes": "générations 8 et 9 (sondage de 2019)",
    "poty_votes": "derrière le 30e de sa région (n° 1 à 890), ou pas encore sorti en 2020 (n° 891 et plus)",
    "poty_rang_generation": "derrière le 30e de sa région (n° 1 à 890), ou pas encore sorti en 2020 (n° 891 et plus)",
    "poty_top30_mondial": "derrière le 30e de sa région (n° 1 à 890), ou pas encore sorti en 2020 (n° 891 et plus)",
})
manquants

,manquants,explication
type2,499,Pokémon à un seul type
type_starter,944,pas un starter
dex_starter,944,pas un starter
reddit_votes,216,générations 8 et 9 (sondage de 2019)
poty_votes,785,"derrière le 30e de sa région (n° 1 à 890), ou ..."
poty_rang_generation,785,"derrière le 30e de sa région (n° 1 à 890), ou ..."
poty_top30_mondial,785,"derrière le 30e de sa région (n° 1 à 890), ou ..."


In [43]:
# chaque manquant correspond-il bien à son explication ?
# un 2e type existe dans PokéAPI pour la forme par défaut de ces Pokémon ?
a_deux_types = default.set_index("species_id")["id"].isin(ptypes.loc[ptypes.slot == 2, "pokemon_id"])
print("type2 vide ⇔ un seul type dans PokéAPI :", (table.type2.isna() == ~a_deux_types.reindex(table.index)).all())
print("starter vide ⇔ pas starter :", (table.type_starter.isna() == ~table.starter).all())
print("Reddit vide ⇔ génération 8 ou 9 :", (table.reddit_votes.isna() == table.generation.isin([8, 9])).all())
print("POTY : les 3 colonnes vides en même temps :", (table[["poty_votes", "poty_rang_generation", "poty_top30_mondial"]].isna().nunique(axis=1) == 1).all())

# POTY : seuls les n° 1 à 890 pouvaient être votés (sondage de février 2020)
vide = table.poty_votes.isna()
print("POTY vide : derrière le 30e (n° ≤ 890) :", (vide & (table.index <= 890)).sum(), "| pas encore sorti (n° > 890) :", (vide & (table.index > 890)).sum())


type2 vide ⇔ un seul type dans PokéAPI : True
starter vide ⇔ pas starter : True
Reddit vide ⇔ génération 8 ou 9 : True
POTY : les 3 colonnes vides en même temps : True
POTY vide : derrière le 30e (n° ≤ 890) : 650 | pas encore sorti (n° > 890) : 135


Il y a 7 colonnes avec des manquants, et **tous s'expliquent** : on a vérifié pour chacune que les cases vides sont exactement celles qu'on attend (un seul type dans PokéAPI, pas starter, générations 8 et 9 pour Reddit, Pokémon non publiés ou pas encore sortis en 2020 pour Pokémon of the Year). On ne remplit donc rien : une case vide veut dire « ne s'applique pas » ou « on ne sait pas », jamais 0.

### 11.3 La cohérence entre les colonnes

In [44]:
stats6 = ["pv", "attaque", "defense", "attaque_spe", "defense_spe", "vitesse"]
controles = {
    "total des stats = somme des 6 stats": (table[stats6].sum(axis=1) == table.total_stats).all(),
    "jamais légendaire et fabuleux à la fois": not (table.legendaire & table.fabuleux).any(),
    "jamais légendaire (ou fabuleux) et starter": not ((table.legendaire | table.fabuleux) & table.starter).any(),
    "jamais bébé et légendaire": not (table.bebe & table.legendaire).any(),
    "jamais deux fois le même type": not (table.type1 == table.type2).any(),
    "les % Johto Times font 100 dans chaque génération": (table.groupby("generation").johto_pct_generation.sum().round(1) == 100).all(),
    "le 1er au Johto Times a le plus haut % de sa génération": (table.loc[table.johto_rang_generation == 1, "johto_pct_generation"].values == table.groupby("generation").johto_pct_generation.max().values).all(),
    "POTY : dans chaque génération, rangs 1, 2, 3… sans trou": table.dropna(subset=["poty_rang_generation"]).groupby("generation").poty_rang_generation.apply(lambda r: sorted(r) == list(range(1, len(r) + 1))).all(),
    "taille, poids et stats > 0": (table[["taille_m", "poids_kg"] + stats6] > 0).all().all(),
}
pd.Series(controles, name="vrai ?")

total des stats = somme des 6 stats                        True
jamais légendaire et fabuleux à la fois                    True
jamais légendaire (ou fabuleux) et starter                 True
jamais bébé et légendaire                                  True
jamais deux fois le même type                              True
les % Johto Times font 100 dans chaque génération          True
le 1er au Johto Times a le plus haut % de sa génération    True
POTY : dans chaque génération, rangs 1, 2, 3… sans trou    True
taille, poids et stats > 0                                 True
Name: vrai ?, dtype: bool

Tous les contrôles de cohérence passent : les stats s'additionnent bien, les catégories (légendaire, fabuleux, bébé, starter) ne se chevauchent pas, et les sources de popularité sont cohérentes avec elles-mêmes (les pourcentages du Johto Times font bien 100 dans chaque génération, les rangs POTY se suivent sans trou dans chaque génération).

### 11.4 Les types de variables

In [45]:
# colonnes entières stockées en nombres à virgule (à cause des cases vides)
a_corriger = ["reddit_votes", "poty_votes", "poty_rang_generation", "dex_starter", "poty_top30_mondial"]
table[a_corriger].dtypes

reddit_votes            float64
poty_votes              float64
poty_rang_generation    float64
dex_starter             float64
poty_top30_mondial      boolean
dtype: object

Ces colonnes sont des **entiers** (des votes, des rangs, un numéro de Pokédex) ou un **oui/non**, mais pandas les a stockées autrement : une colonne d'entiers classique ne peut pas contenir de case vide, alors pandas la passe en nombres à virgule (`710.0` au lieu de `710`). On utilise à la place les types « nullables » de pandas, qui acceptent les cases vides : `Int64` pour les entiers et `boolean` pour les oui/non.

In [46]:
for col in ["reddit_votes", "poty_votes", "poty_rang_generation", "dex_starter"]:
    assert (table[col].dropna() % 1 == 0).all(), col  # on vérifie qu'on ne perd aucune décimale
    table[col] = table[col].astype("Int64")
table["poty_top30_mondial"] = table["poty_top30_mondial"].astype("boolean")
table[a_corriger].dtypes

reddit_votes              Int64
poty_votes                Int64
poty_rang_generation      Int64
dex_starter               Int64
poty_top30_mondial      boolean
dtype: object

Les votes, rangs et numéros sont maintenant des entiers, et les valeurs n'ont pas changé (on a vérifié qu'aucune n'avait de décimale).

**Attention en relisant le CSV** : un fichier CSV ne garde pas les types. `pd.read_csv` repasse ces colonnes en nombres à virgule, et `poty_top30_mondial` en texte. Pour retrouver les bons types :

```python
table = pd.read_csv(PROC / "table_maitre.csv", index_col="dex", dtype={
    "reddit_votes": "Int64", "poty_votes": "Int64", "poty_rang_generation": "Int64",
    "dex_starter": "Int64", "poty_top30_mondial": "boolean"})
```

### 11.5 Les valeurs extrêmes

In [47]:
# règle de Tukey : en dehors de [Q1 − 1,5 × IQR ; Q3 + 1,5 × IQR]
# taille et poids sont très asymétriques (de 0,1 à 999,9 kg) : on applique la règle sur leur logarithme
import numpy as np

continues = ["taille_m", "poids_kg"] + stats6 + ["total_stats", "taux_capture"]
lignes = []
for col in continues:
    x = np.log10(table[col]) if col in ("taille_m", "poids_kg") else table[col]
    q1, q3 = x.quantile([0.25, 0.75])
    bas, haut = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    ext = table[(x < bas) | (x > haut)].sort_values(col)
    lignes.append({
        "variable": col, "extrêmes": len(ext),
        "plus bas": f"{ext.nom_fr.iloc[0]} ({ext[col].iloc[0]})" if len(ext) else "",
        "plus haut": f"{ext.nom_fr.iloc[-1]} ({ext[col].iloc[-1]})" if len(ext) else "",
    })
pd.DataFrame(lignes).set_index("variable")

,extrêmes,plus bas,plus haut
variable,,,
taille_m,6,Onix (8.8),Éthernatos (20.0)
poids_kg,16,Fantominus (0.1),Pêchaminus (0.3)
pv,18,Grodoudou (140),Leuphorie (255)
attaque,1,Katagami (181),Katagami (181)
defense,11,Prédastérie (152),Caratroc (230)
attaque_spe,1,Câblifère (173),Câblifère (173)
defense_spe,11,Prédastérie (142),Caratroc (230)
vitesse,2,Ninjask (160),Regieleki (200)
total_stats,0,,


La règle de Tukey signale entre 0 et 18 Pokémon selon la variable. On a regardé qui ils sont : **ce sont de vrais Pokémon, pas des erreurs**. Les extrêmes de taille sont tous des géants (d'Onix, 8,8 m, à Éthernatos, 20 m), ceux de poids des poids plume (de 0,1 à 0,3 kg, comme Fantominus, un fantôme fait de gaz), et en stats ce sont des Pokémon connus pour ça (Leuphorie et ses 255 PV, Caratroc et sa défense de 230).

On les **garde tous** : les enlever reviendrait à enlever des Pokémon de l'étude. Pour que les géants ne pèsent pas trop dans les calculs, on passera la taille et le poids en logarithme quand c'est nécessaire (c'est ce que fait le notebook 06).

### 11.6 Le poids plafonné à 999,9 kg

In [48]:
table.sort_values("poids_kg", ascending=False)[["nom_fr", "taille_m", "poids_kg", "legendaire"]].head(5)

,nom_fr,taille_m,poids_kg,legendaire
dex,,,,
790,Cosmovum,0.1,999.9,True
797,Bamboiselle,9.2,999.9,False
383,Groudon,3.5,950.0,True
890,Éthernatos,20.0,950.0,True
750,Bourrinos,2.5,920.0,False


Deux Pokémon pèsent **exactement 999,9 kg** : Cosmovum et Bamboiselle. Ce n'est pas un hasard : c'est le poids maximum que les jeux affichent. Leur vrai poids est inconnu (pour Cosmovum, 999,9 kg pour 10 cm, ce serait de toute façon une densité absurde).

La règle de Tukey ne les voit pas (sur l'échelle logarithmique, 999,9 kg reste dans la norme des gros Pokémon), d'où ce contrôle à part. On garde la valeur, mais on ajoute une colonne **`poids_plafonne`** pour pouvoir les écarter des calculs où le poids exact compte (la densité, dans l'onglet Biologie).

In [49]:
table["poids_plafonne"] = table["poids_kg"] >= 999.9
table.loc[table.poids_plafonne, ["nom_fr", "poids_kg", "poids_plafonne"]]

,nom_fr,poids_kg,poids_plafonne
dex,,,
790,Cosmovum,999.9,True
797,Bamboiselle,999.9,True


### 11.7 La mise à l'échelle

On **ne met pas les variables à l'échelle dans la table maître** : on exporte les valeurs brutes, dans leurs unités (mètres, kilos, points de stats), pour qu'elles restent lisibles. Chaque analyse qui en a besoin fait sa propre mise à l'échelle, adaptée à sa méthode : logarithme de la taille et du poids et standardisation pour l'ACP (notebook 06), variables standardisées pour le réseau de similarité (notebook 07).

## 12. Export

In [50]:
table.to_csv(OUT / "table_maitre.csv")
print("écrit :", OUT / "table_maitre.csv", table.shape)

écrit : ../data/processed/table_maitre.csv (1025, 35)
